In [2]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
diabetes_rf = joblib.load('../output/diabetes_rf_model.joblib')
heart_lasso = joblib.load('../output/heart_disease_lasso_model.joblib')
stroke_lasso = joblib.load('../output/stroke_model.joblib')

In [5]:
DIABETES_FEATURES = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
HEART_FEATURES = ["Age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"]

# Stroke features require looking at your encoder, but I can extract the count from the model
stroke_coef = stroke_lasso.coef_[0]

In [6]:
# Analyze Random Forest model for Diabetes risk prediction
rf_importances = diabetes_rf.feature_importances_
df_rf = pd.DataFrame({'Feature': DIABETES_FEATURES, 'Importance': rf_importances})
df_rf = df_rf.sort_values('Importance', ascending=False)
print("Top 3 Attack Vectors (by Gini Importance):")
print(df_rf.head(3).to_string(index=False))

Top 3 Attack Vectors (by Gini Importance):
Feature  Importance
Glucose    0.264336
    BMI    0.159328
    Age    0.154988


In [10]:
# Analyze Lasso LR model (Heart Disease - C=0.1)
print("\nHeart Disease (Lasso LR, C=0.1)")
heart_coef = heart_lasso.coef_[0]
df_heart = pd.DataFrame({'Feature': HEART_FEATURES, 'Coefficient': heart_coef, 'Absolute_Weight': np.abs(heart_coef)})
df_heart = df_heart.sort_values('Absolute_Weight', ascending=False)
zeroed_heart = (df_heart['Coefficient'] == 0).sum()
print(f"Total Features: {len(HEART_FEATURES)}")
print(f"Features Zeroed out by Lasso (Invulnerable to attack): {zeroed_heart}")
print("Top 3 Attack Vectors (by absolute weight):")
print(df_heart[df_heart['Coefficient'] != 0].head(3)[['Feature', 'Coefficient']].to_string(index=False))


Heart Disease (Lasso LR, C=0.1)
Total Features: 13
Features Zeroed out by Lasso (Invulnerable to attack): 2
Top 3 Attack Vectors (by absolute weight):
Feature  Coefficient
     cp     0.702687
oldpeak    -0.642355
     ca    -0.620334


In [11]:
# Analyze Lasso LR (Stroke - C=0.01)
print("\nStroke (Lasso LR, C=0.01)")
zeroed_stroke = (stroke_coef == 0).sum()
print(f"Total Encoded Features: {len(stroke_coef)}")
print(f"Features Zeroed out by Lasso (Invulnerable to attack): {zeroed_stroke}")
print(f"Sparsity Ratio: {(zeroed_stroke/len(stroke_coef))*100:.1f}% off the attack surface.")


Stroke (Lasso LR, C=0.01)
Total Encoded Features: 14
Features Zeroed out by Lasso (Invulnerable to attack): 9
Sparsity Ratio: 64.3% off the attack surface.


Hypothesis Confirmed. The stricter C=0.01 penalty on the stroke model severely restricts the attacker's surface area compared to the heart model.